In [2]:

import re
from collections import defaultdict

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT = {'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,'risorse':5,'stabilita':5,
           'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
           'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,
           'risorse_russia':5,'influenza_militare_russia':5,'veto_onu_russia':2,'stabilita_economica_russia':5,'stabilita_russia':5,
           'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,'stabilita_rotte_cina':5,'stabilita_cina':5,
           'risorse_europa':5,'influenza_diplomatica_europa':5,'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5}

WEIGHTS_IRAN = {'nucleare':+3,'sanzioni':-3,'opinione':-2,'defcon':-1,'risorse':+2,'stabilita':+2,
                'risorse_iran':+2,'forze_militari_iran':+2,'tecnologia_nucleare_iran':+3,'stabilita_iran':+2,
                'stabilita_coalizione':-1,'forze_militari_iran':+2}
WEIGHTS_COAL = {'nucleare':-3,'sanzioni':+3,'opinione':+2,'defcon':+1,'risorse':+2,'stabilita':+2,
                'risorse_coalizione':+2,'tecnologia_avanzata_coalizione':+2,'supporto_pubblico_coalizione':+2,
                'influenza_diplomatica_coalizione':+2,'tecnologia_nucleare_iran':-3,'forze_militari_iran':-2}

card_re = re.compile(r"card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)',\s*card_type:'([^']+)'.*?effects:\{([^}]+)\}", re.DOTALL)

faz_scores = defaultdict(list)
for m in card_re.finditer(src):
    cid, cname, faz, ctype, eff_str = m.groups()
    score_iran = 0; score_coal = 0
    for t in list(DEFAULT.keys()):
        fn = re.search(rf'{t}\s*:\s*\(v[^)]*\)\s*=>\s*([^,}}]+)', eff_str)
        if fn:
            try:
                v = float(eval(f"(lambda v:{fn.group(1).strip().rstrip(',')})(DEFAULT['{t}'])", {'DEFAULT':DEFAULT}))
                score_iran += v * WEIGHTS_IRAN.get(t, 0)
                score_coal += v * WEIGHTS_COAL.get(t, 0)
            except: pass
    faz_scores[faz].append((cid, cname, score_iran, score_coal))

print("=== PUNTEGGI MEDI PRIMA DEL BILANCIAMENTO ===")
for faz in ['Iran','Coalizione','Russia','Cina','Europa','Neutrale']:
    cards = faz_scores[faz]
    if not cards: continue
    avg_i = sum(c[2] for c in cards)/len(cards)
    avg_c = sum(c[3] for c in cards)/len(cards)
    print(f"{faz}: avg score Iran={avg_i:.2f} Coal={avg_c:.2f} ({len(cards)} carte)")

coal_cards = sorted(faz_scores['Coalizione'], key=lambda x: x[3], reverse=True)
print("\nTop 10 carte Coalizione più forti:")
for c in coal_cards[:10]:
    print(f"  {c[0]} '{c[1]}': score_coal={c[3]:.1f} score_iran={c[2]:.1f}")

iran_cards = sorted(faz_scores['Iran'], key=lambda x: x[2])
print("\nTop 10 carte Iran più deboli:")
for c in iran_cards[:10]:
    print(f"  {c[0]} '{c[1]}': score_iran={c[2]:.1f} score_coal={c[3]:.1f}")


=== PUNTEGGI MEDI PRIMA DEL BILANCIAMENTO ===
Iran: avg score Iran=3.66 Coal=-2.33 (70 carte)
Coalizione: avg score Iran=-7.57 Coal=7.97 (70 carte)
Russia: avg score Iran=4.00 Coal=-0.33 (58 carte)
Cina: avg score Iran=3.86 Coal=0.93 (58 carte)
Europa: avg score Iran=-2.67 Coal=5.90 (58 carte)
Neutrale: avg score Iran=-0.77 Coal=-1.63 (30 carte)

Top 10 carte Coalizione più forti:
  NC40 'Proposta Pace Definitiva': score_coal=25.0 score_iran=-23.0
  NC04 'Proposta JCPOA 3.0': score_coal=23.0 score_iran=-21.0
  SE_C04 'Accordo Nucleare Last Minute (spec.)': score_coal=23.0 score_iran=-21.0
  NC01 'Accordo AIEA Mar 2026': score_coal=17.0 score_iran=-15.0
  C015 'Operazione Freedom': score_coal=15.0 score_iran=-7.0
  SE_C02 'Operazione Mossad (spec.)': score_coal=14.0 score_iran=-12.0
  NC02 'Risoluzione ONU 2891': score_coal=13.0 score_iran=-11.0
  NC31 'Embargo Armi Iran Rafforzato': score_coal=13.0 score_iran=-9.0
  C004 'Cyber Attack Stuxnet 2.0': score_coal=12.0 score_iran=-10.0
  C0

In [5]:

import re

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

changes = 0

# 1. Limita sanzioni Coalizione: max +2 per carta (non +3)
# e nucleare Coalizione max -2 (non -3)
lines = src.split('\n')
new_lines = []
in_coalizione = False
san_nuc_fixes = 0
for i, line in enumerate(lines):
    if "faction:'Coalizione'" in line:
        in_coalizione = True
    elif "faction:'" in line and "Coalizione" not in line:
        in_coalizione = False
    
    if in_coalizione and 'sanzioni:(v)=>3' in line:
        line = line.replace('sanzioni:(v)=>3', 'sanzioni:(v)=>2')
        san_nuc_fixes += 1; changes += 1
    if in_coalizione and 'sanzioni:(v)=>+3' in line:
        line = line.replace('sanzioni:(v)=>+3', 'sanzioni:(v)=>2')
        san_nuc_fixes += 1; changes += 1
    if in_coalizione and 'nucleare:(v)=>-3' in line:
        line = line.replace('nucleare:(v)=>-3', 'nucleare:(v)=>-2')
        san_nuc_fixes += 1; changes += 1
    new_lines.append(line)

src = '\n'.join(new_lines)

# 2. Potenzia carte Iran Militare senza nucleare: aggiungi nucleare:(v)=>1
potenziati = 0
card_re = re.compile(
    r"(\{\s*card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'Iran',\s*card_type:'Militare'[^}]*effects:\{)([^}]+)(\})",
    re.DOTALL
)
# Trova tutte le carte Iran Militare senza nucleare, ordina per score_iran ascendente (deboli prima)
# Per semplicità usiamo l'ordine di apparizione e saltiamo se già hanno nucleare
matches_to_patch = []
for m in card_re.finditer(src):
    eff = m.group(4)
    if 'nucleare' not in eff:
        matches_to_patch.append(m)

# Patch al contrario (dalla fine) per non spostare gli offset
for m in reversed(matches_to_patch[:8]):
    eff = m.group(4)
    new_eff = eff.rstrip() + ',\n              nucleare:(v)=>v+1'
    src = src[:m.start(4)] + new_eff + src[m.end(4):]
    potenziati += 1
    changes += 1

# 3. Aggiungi costo risorse_coalizione:(v)=>v-1 alle carte Coalizione con nucleare-2 o sanzioni+2
#    ma solo se non hanno già un costo risorse
coalizione_costo = 0
coal_re = re.compile(
    r"(\{\s*card_id:'N?C\d+[^']*',\s*card_name:'[^']*',\s*faction:'Coalizione'[^}]*effects:\{)([^}]+)(\})",
    re.DOTALL
)
# Anche qui patch al contrario
matches_coal = []
for m in coal_re.finditer(src):
    eff = m.group(2)
    has_nuc_minus2 = 'nucleare:(v)=>-2' in eff
    has_san_plus2  = 'sanzioni:(v)=>2' in eff or 'sanzioni:(v)=>+2' in eff
    has_risorse    = 'risorse' in eff
    if (has_nuc_minus2 or has_san_plus2) and not has_risorse and coalizione_costo < 10:
        matches_coal.append(m)
        coalizione_costo += 1

for m in reversed(matches_coal):
    eff = m.group(2)
    new_eff = eff.rstrip() + ',\n              risorse_coalizione:(v)=>v-1'
    src = src[:m.start(2)] + new_eff + src[m.end(2):]
    changes += 1

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts', 'w') as f:
    f.write(src)

print(f"Modifiche totali: {changes}")
print(f"  Sanzioni/nucleare Coalizione ridotti: {san_nuc_fixes}")
print(f"  Carte Iran potenziate (+nucleare): {potenziati}")
print(f"  Costo risorse aggiunto a Coalizione: {coalizione_costo}")


Modifiche totali: 19
  Sanzioni/nucleare Coalizione ridotti: 6
  Carte Iran potenziate (+nucleare): 8
  Costo risorse aggiunto a Coalizione: 5


In [8]:

import re
from collections import defaultdict

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT = {'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,'risorse':5,'stabilita':5,
           'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
           'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,
           'risorse_russia':5,'influenza_militare_russia':5,'veto_onu_russia':2,'stabilita_economica_russia':5,'stabilita_russia':5,
           'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,'stabilita_rotte_cina':5,'stabilita_cina':5,
           'risorse_europa':5,'influenza_diplomatica_europa':5,'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5}

WEIGHTS_IRAN = {'nucleare':+3,'sanzioni':-3,'opinione':-2,'defcon':-1,'risorse':+2,'stabilita':+2,
                'risorse_iran':+2,'forze_militari_iran':+2,'tecnologia_nucleare_iran':+3,'stabilita_iran':+2,
                'stabilita_coalizione':-1,'forze_militari_iran':+2}
WEIGHTS_COAL = {'nucleare':-3,'sanzioni':+3,'opinione':+2,'defcon':+1,'risorse':+2,'stabilita':+2,
                'risorse_coalizione':+2,'tecnologia_avanzata_coalizione':+2,'supporto_pubblico_coalizione':+2,
                'influenza_diplomatica_coalizione':+2,'tecnologia_nucleare_iran':-3,'forze_militari_iran':-2}

card_re = re.compile(r"card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)',\s*card_type:'([^']+)'.*?effects:\{([^}]+)\}", re.DOTALL)

faz_scores = defaultdict(list)
for m in card_re.finditer(src):
    cid, cname, faz, ctype, eff_str = m.groups()
    score_iran = 0; score_coal = 0
    for t in list(DEFAULT.keys()):
        fn = re.search(rf'{t}\s*:\s*\(v[^)]*\)\s*=>\s*([^,}}]+)', eff_str)
        if fn:
            try:
                v = float(eval(f"(lambda v:{fn.group(1).strip().rstrip(',')})(DEFAULT['{t}'])", {'DEFAULT':DEFAULT}))
                score_iran += v * WEIGHTS_IRAN.get(t, 0)
                score_coal += v * WEIGHTS_COAL.get(t, 0)
            except: pass
    faz_scores[faz].append((cid, cname, score_iran, score_coal))

print("=== PUNTEGGI MEDI DOPO IL BILANCIAMENTO ===")
for faz in ['Iran','Coalizione','Russia','Cina','Europa','Neutrale']:
    cards = faz_scores[faz]
    if not cards: continue
    avg_i = sum(c[2] for c in cards)/len(cards)
    avg_c = sum(c[3] for c in cards)/len(cards)
    print(f"{faz}: avg score Iran={avg_i:.2f} Coal={avg_c:.2f} ({len(cards)} carte)")

print("\nTop 5 carte Coalizione più forti (post-fix):")
coal_cards = sorted(faz_scores['Coalizione'], key=lambda x: x[3], reverse=True)
for c in coal_cards[:5]:
    print(f"  {c[0]} '{c[1]}': score_coal={c[3]:.1f}")


=== PUNTEGGI MEDI DOPO IL BILANCIAMENTO ===
Iran: avg score Iran=5.71 Coal=-4.39 (70 carte)
Coalizione: avg score Iran=-7.31 Coal=8.29 (70 carte)
Russia: avg score Iran=4.00 Coal=-0.33 (58 carte)
Cina: avg score Iran=3.86 Coal=0.93 (58 carte)
Europa: avg score Iran=-2.67 Coal=5.90 (58 carte)
Neutrale: avg score Iran=-0.77 Coal=-1.63 (30 carte)

Top 5 carte Coalizione più forti (post-fix):
  NC40 'Proposta Pace Definitiva': score_coal=30.0
  NC04 'Proposta JCPOA 3.0': score_coal=28.0
  NC01 'Accordo AIEA Mar 2026': score_coal=25.0
  NC05 'Summit G7 Iran Mar 2026': score_coal=20.0
  SE_C04 'Accordo Nucleare Last Minute (spec.)': score_coal=20.0
